In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import k3d

In [ ]:
f0 = np.load('/home/yousen/Documents/NDLAr2x2/larnd-sim/waveforms_default2.npz')
f1 = np.load('/home/yousen/Documents/NDLAr2x2/larnd-sim/waveforms_full3.npz')

ftred = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_neighboring_pixels_20250623/tred_for_larndsim_2tks.npz')
ftred = np.load('/home/yousen/Public/ndlar_shared/data/larndsim_neighboring_pixels_20250623/tred_for_larndsim_2tks_truncate_tref_20250625.npz')

In [ ]:
np.sum(ftred['current_tpc0_batch0'][:,:,:]), np.sum(ftred['current_tpc0_batch0'][:,:,:])*1E3/2442891

In [ ]:
f0.files

In [ ]:
f0['max_radius'], f0['max_pixels'], f0['max_neighboring_pixels'], f0['pixel_index_from_segment'], f0['unique_pixel_index'].shape

In [ ]:
def id2pixel(pid):
    """
    Convert the unique pixel identifer to an x,y,plane tuple

    Args:
        pid (int): unique pixel identifier
    Returns:
        tuple: number of pixel pitches in x-dimension,
            number of pixel pitches in y-dimension,
            pixel plane number
    """
    npixels = (2*70, 4*70)
    return (pid % npixels[0], (pid // npixels[0]) % npixels[1],
            (pid // (npixels[0] * npixels[1])))


def id2pxlcoord(pid):
    """
    Convert the unique pixel identifer to an x,y,plane tuple

    Args:
        pid (int): unique pixel identifier
    Returns:
        tuple: number of pixel pitches in x-dimension,
            number of pixel pitches in y-dimension,
            pixel plane number
    """
    npixels = (2*70, 4*70)
    zi, yi, pi = (pid % npixels[0], (pid // npixels[0]) % npixels[1],
            (pid // (npixels[0] * npixels[1])))
    if np.any(pi != 1):
        raise NotImplementedError()
    return 2.462 + (zi+0.5) * 0.4434, -62.076 + (yi+0.5) * 0.4434

def ind2coord(iz, iy):
    return 2.462 + (iz+0.5) * 0.4434, -62.076 + (iy+0.5) * 0.4434

In [ ]:
z0, y0, planes0 = id2pixel(f0['unique_pixel_index'])
z1, y1, planes1 = id2pixel(f1['unique_pixel_index'])

z0f, y0f = id2pxlcoord(f0['unique_pixel_index'])
z1f ,y1f = id2pxlcoord(f1['unique_pixel_index'])

planes0, planes1

In [ ]:
plt.plot(z1[0], y1[0], 'o', label='full')
plt.plot(z0[0], y0[0], 'o', label='default')
plt.legend()

In [ ]:
plt.scatter(z1f[0], y1f[0], label='full')
plt.scatter(z0f[0], y0f[0], label='default')
for pid, c in zip((13, 2212), ('r', 'g')):
    m = np.abs(f1['segment_info'][0]['pdg_id']) == pid
    x = ( *(f1['segment_info'][0]['x_start'][m]), f1['segment_info'][0]['x_end'][m][-1])
    print(pid, f1['segment_info'][0]['x_start'][m], f1['segment_info'][0]['x_end'][m])
    print(pid, f1['segment_info'][0]['y_start'][m], f1['segment_info'][0]['y_end'][m])
    y = ( *(f1['segment_info'][0]['y_start'][m]), f1['segment_info'][0]['y_end'][m][-1])
    plt.plot(x, y, label=f'abs(pid) == {pid}', marker='*', markeredgecolor=c, color=c)
plt.xlabel('x (beam direction) [cm]')
plt.ylabel('y (vertical direction) [cm]')
plt.legend()

In [ ]:
np.sum(f0['hit_charges']), np.sum(f1['hit_charges']), np.sum(f0['pixel_waveform'][0])* 0.1 , np.sum(f1['pixel_waveform'][0]) * 0.1 , np.sum(f0['segment_info']['n_electrons']), np.sum(f1['segment_info']['n_electrons'])

In [ ]:
np.sum(f0['hit_charges'])/2442891, np.sum(f1['hit_charges'])/2442891, np.sum(f0['pixel_waveform'][0])* 0.1/2442891 , np.sum(f1['pixel_waveform'][0]) * 0.1/2442891

In [ ]:
np.nonzero(f0['pixel_waveform'][0])

In [ ]:
f0['pixel_waveform'][0].shape

In [ ]:
np.unique(np.nonzero(f0['pixel_waveform'][0])[0]).shape

In [ ]:
# for ipix in range(f0['hit_charges'].shape[1]):
#     if any(f0['hit_charges'][0,ipix]):
#         print(f0['hit_charges'][0,ipix])
np.nonzero(f0['hit_charges'][0])

In [ ]:
totq1 = np.sum(f1['hit_charges'][0], axis=-1)
pxl_inds = np.nonzero(totq1)[0]
plt.scatter(x=z1[0][pxl_inds], y=y1[0][pxl_inds], c=totq1[pxl_inds])
plt.colorbar()

In [ ]:
totq1 = np.sum(f1['hit_charges'][0], axis=-1)
pxl_inds = np.nonzero(totq1)[0]
plt.scatter(x=z1f[0][pxl_inds], y=y1f[0][pxl_inds], c=totq1[pxl_inds])
plt.colorbar()

for pid in (13, 2212):
    m = np.abs(f1['segment_info'][0]['pdg_id']) == pid
    x = ( *(f1['segment_info'][0]['x_start'][m]), f1['segment_info'][0]['x_end'][m][-1])
    print(pid, f1['segment_info'][0]['x_start'][m], f1['segment_info'][0]['x_end'][m])
    print(pid, f1['segment_info'][0]['y_start'][m], f1['segment_info'][0]['y_end'][m])
    y = ( *(f1['segment_info'][0]['y_start'][m]), f1['segment_info'][0]['y_end'][m][-1])
    plt.plot(x, y, label=f'pid == {pid}', marker='o')
plt.legend()
plt.title('from [-5, 5] neighboring pixels')
plt.xlabel('beam direction [cm]')
plt.ylabel('vertical direction [cm]')

In [ ]:
totq = np.sum(f0['hit_charges'][0], axis=-1)
pxl_inds = np.nonzero(totq)[0]
plt.scatter(x=z0[0][pxl_inds], y=y0[0][pxl_inds], c=totq[pxl_inds])
plt.colorbar()

In [ ]:
totq = np.sum(f0['hit_charges'][0], axis=-1)
pxl_inds = np.nonzero(totq)[0]
plt.scatter(x=z0f[0][pxl_inds], y=y0f[0][pxl_inds], c=totq[pxl_inds])
plt.colorbar()
for pid in (13, 2212):
    m = np.abs(f0['segment_info'][0]['pdg_id']) == pid
    x = ( *(f0['segment_info'][0]['x_start'][m]), f0['segment_info'][0]['x_end'][m][-1])
    print(pid, f0['segment_info'][0]['x_start'][m], f0['segment_info'][0]['x_end'][m])
    y = ( *(f0['segment_info'][0]['y_start'][m]), f0['segment_info'][0]['y_end'][m][-1])
    plt.plot(x, y, label=f'abs(pid) == {pid}', marker='o')
plt.legend()
plt.title('from default neighboring pixels')
plt.xlabel('beam direction [cm]')
plt.ylabel('vertical direction [cm]')

In [ ]:
def plot_wf(iz, iy):
    pzm0 = z0[0] == iz
    pym0 = y0[0] == iy
    wf0 = f0['pixel_waveform'][0][ pzm0 & pym0]
    if len(wf0):
        wf0 = wf0[0]

    pzm1 = z1[0] == iz
    pym1 = y1[0] == iy
    wf1 = f1['pixel_waveform'][0][ pzm1 & pym1][0]
    if len(wf0):
        plt.plot(wf0, label=f'default sum {np.sum(wf0)*0.1}')
    plt.plot(wf1, label=f'full, sum {np.sum(wf1)*0.1}', linestyle='--')
    plt.legend()
    plt.title(f'(ix, iy) = {(iz, iy)}, (x, y) = ({z1f[0][pzm1][0]:.3f},{y1f[0][pym1][0]:.3f})')

In [ ]:
plot_wf(45, 151)

In [ ]:
plot_wf(62, 152)

In [ ]:
plot_wf(45, 152)

In [ ]:
plot_wf(45, 155)

In [ ]:
plot_wf(55, 154)

In [ ]:
plot_wf(55, 155)

In [ ]:
plot_wf(55, 153)

In [ ]:
def plt_3views(f):
    fig, axes = plt.subplots(1, 3, figsize=(3*6, 6))
    pids = np.unique(f['pdg_id'])
    for pid in pids:
        m = pid == f['pdg_id']
        x = ( *f['x_start'][m], f['x_end'][m][-1])
        y = ( *f['y_start'][m], f['y_end'][m][-1])
        z = ( *f['z_start'][m], f['z_end'][m][-1])
        axes[0].plot(x, y, label=f'pdg_id: {f["pdg_id"][m][0]}', marker='o')
        axes[1].plot(y, z, label=f'pdg_id: {f["pdg_id"][m][0]}', marker='o')
        axes[2].plot(z, x, label=f'pdg_id: {f["pdg_id"][m][0]}', marker='o')
    axes[0].legend()
    axes[1].legend()
    axes[2].legend()
    axes[0].set_xlabel('x (beam direction) [cm]')
    axes[0].set_ylabel('y (vertical direction) [cm]')
    axes[1].set_xlabel('y (vertical direction) [cm]')
    axes[1].set_ylabel('z (drift direction) [cm]')
    axes[2].set_xlabel('z (drift direction) [cm]')
    axes[2].set_ylabel('x (beam direction) [cm]')

In [ ]:
plt_3views(f0['segment_info'][0])

In [ ]:
def plt_k3d(f, plot):
    pids = np.unique(f['pdg_id'])
    for pid in pids:
        m = pid == f['pdg_id']
        x = ( *f['x_start'][m], f['x_end'][m][-1])
        y = ( *f['y_start'][m], f['y_end'][m][-1])
        z = ( *f['z_start'][m], f['z_end'][m][-1])
        points = np.vstack([x,y,z]).T.flatten()
        line = k3d.line(points)
        plot += line

In [ ]:
plot = k3d.plot()
plt_k3d(f0['segment_info'][0], plot)

In [ ]:
plot.display()

In [ ]:
res = np.load("/home/yousen/Public/ndlar_shared/data/responses/response_44_v2a_full.npz")
# res = np.load("/home/yousen/Documents/NDLAr2x2/larnd-sim/waveforms_default3.npz")

In [ ]:
np.sum(res['response'][:,:,2000:]) * 0.05 

In [ ]:
def check_seg(iz, iy):
    iz0, iy0, _ = id2pixel(f0['neighboring_pixels'])
    pzm0 = iz0[0] == iz
    pym0 = iy0[0] == iy
    m0 = (pzm0 & pym0)
    wf0 = f0['start_end'][0, :, :][m0]

    iz1, iy1, _ = id2pixel(f1['neighboring_pixels'])
    pzm1 = iz1[0] == iz
    pym1 = iy1[0] == iy
    m1 = (pzm1 & pym1)
    wf1 = f1['start_end'][0, :, :][m1]

    print('default', wf0.shape, wf0)
    print('full', wf1.shape, wf1)

In [ ]:
check_seg(45, 155)

In [ ]:
2000*0.16*0.05

In [ ]:
def plot_selected_segments(iz, iy):
    fig, ax = plt.subplots(1,1,figsize=(8,8))
    for pid in (13, 2212):
        m = np.abs(f0['segment_info'][0]['pdg_id']) == pid
        x = ( *(f0['segment_info'][0]['x_start'][m]), f0['segment_info'][0]['x_end'][m][-1])
        y = ( *(f0['segment_info'][0]['y_start'][m]), f0['segment_info'][0]['y_end'][m][-1])
        plt.plot(x, y, label=f'abs(pid) == {pid}', marker='o')
    plt.xlabel('beam direction [cm]')
    plt.ylabel('vertical direction [cm]')
    iz0, iy0, _ = id2pixel(f0['neighboring_pixels'])
    pzm0 = iz0[0] == iz
    pym0 = iy0[0] == iy
    m0 = (pzm0 & pym0)
    wf0 = f0['start_end'][0, :, :][m0]

    iz1, iy1, _ = id2pixel(f1['neighboring_pixels'])
    pzm1 = iz1[0] == iz
    pym1 = iy1[0] == iy
    m1 = (pzm1 & pym1)
    wf1 = f1['start_end'][0, :, :][m1]
    
    x, y = ind2coord(iz, iy)
    plt.plot([x,], [y,], 'o', label='pixel of interest')
    for se in wf0:
        xs = se[[0,0+3]]
        ys = se[[1,1+3]]
        plt.plot(xs, ys, label='default', linestyle='--', linewidth=3)
    for se in wf1:
        xs = se[[0,0+3]]
        ys = se[[1,1+3]]
        plt.plot(xs, ys, label='full', linestyle='-.', linewidth=2)
    plt.legend()
    plt.title(f'Pixel ind ({iz}, {iy}) --> coord ({x:.3f}, {y:.3f})')

    print('default', wf0.shape, wf0)
    print('full', wf1.shape, wf1)
    d1 = wf1[:,:2]-np.array([[x,y]])
    d12 = wf1[:,3:5]-np.array([[x,y]])
    print(np.sqrt(d1[:,0]**2+d1[:,1]**2))
    print(np.sqrt(d12[:,0]**2+d12[:,1]**2))
    print(np.sqrt((x-25.)**2 + (y-5.1)**2))
    print(0.4434*4.5*1.414)

In [ ]:
plot_selected_segments(45, 155)

In [ ]:
def sum_res_charges(d):
    t = abs(d-33.34125)/0.1596452482154287
    # print(t, np.ceil(t/0.05).astype(int))
    q = res['response'][:,:, np.ceil(t/0.05).astype(int):]
    q = np.sum(q) * 0.05
    return q

In [ ]:
print(sum_res_charges(15.5), sum_res_charges(15.5)/25)
print(sum_res_charges(13.5), sum_res_charges(13.5)/25)
print(sum_res_charges(33), sum_res_charges(33)/25)

In [ ]:
f1['pixel_waveform'][0].shape

In [ ]:
cd, cl = ftred['current_tpc0_batch0'], ftred['current_tpc0_batch0_location']
ciy = []
ciz = []
cdiff = []
for i in range(cd.shape[0]):
    m = (cl[i,0] == y1[0]) & (cl[i,1] == z1[0])
    nvalid = np.sum(m)
    if nvalid != 1:
        print(np.sum(cd[i])*1E3, f1['pixel_waveform'][0][m], np.sum(f1['pixel_waveform'][0][m]), cl[i,0], cl[i,1])
    ciy.append(cl[i,0])
    ciz.append(cl[i,1])
    cdiff.append(np.sum(f1['pixel_waveform'][0][m]) * 0.1-np.sum(cd[i])*1E3)
plt.scatter(ciz, ciy, c=np.abs(cdiff))
cbar = plt.colorbar()
cbar.set_label('abs(dQ) per pixel (larndsim - TRED)')
plt.title('selected pixels from TRED')

In [ ]:
cd, cl = ftred['current_tpc0_batch0'], ftred['current_tpc0_batch0_location']
ciy = []
ciz = []
cdiff = []
for i in range(cd.shape[0]):
    m = (cl[i,0] == y1[0]) & (cl[i,1] == z1[0])
    nvalid = np.sum(m)
    if nvalid != 1:
        print(np.sum(cd[i])*1E3, f1['pixel_waveform'][0][m], np.sum(f1['pixel_waveform'][0][m]), cl[i,0], cl[i,1])
    ciy.append(cl[i,0])
    ciz.append(cl[i,1])
    cdiff.append(np.sum(f1['pixel_waveform'][0][m]) * 0.1-np.sum(cd[i])*1E3)
plt.scatter(ciz, ciy, c=cdiff)
cbar = plt.colorbar()
cbar.set_label('dQ per pixel (larndsim - TRED)')
plt.title('selected pixels from TRED')

In [ ]:
np.sum(ftred['hits_tpc0_batch0'][:,-1])

In [ ]:
np.sum(ftred['hits_tpc0_batch0'][:,-1])*1E3/2442891

In [ ]:
upxl, ind, inv = np.unique(ftred['hits_tpc0_batch0_location'][:,:2], axis=0, return_index=True, return_inverse=True)
totq_tred = np.zeros(len(upxl))
for i in range(len(upxl)):
    m = i == inv
    totq_tred[i] = np.sum(ftred['hits_tpc0_batch0'][:,-1][m])
locs = ftred['hits_tpc0_batch0'][:,[1,2]][ind]
# print(len(locs), len(totq_tred))
plt.scatter(x=locs[:,1], y=locs[:,0], c=np.round(totq_tred*1E3))
plt.colorbar()

for pid in (13, 2212):
    m = np.abs(f1['segment_info'][0]['pdg_id']) == pid
    x = ( *(f1['segment_info'][0]['x_start'][m]), f1['segment_info'][0]['x_end'][m][-1])
    print(pid, f1['segment_info'][0]['x_start'][m], f1['segment_info'][0]['x_end'][m])
    print(pid, f1['segment_info'][0]['y_start'][m], f1['segment_info'][0]['y_end'][m])
    y = ( *(f1['segment_info'][0]['y_start'][m]), f1['segment_info'][0]['y_end'][m][-1])
    plt.plot(x, y, label=f'pid == {pid}', marker='o')
plt.legend()
plt.title('from tred')
plt.xlabel('beam direction [cm]')
plt.ylabel('vertical direction [cm]')